# Configuración del entorno en Databricks CE

- Runtime: Databricks CE (Spark 4.1.0, Scala 2.12)
- Núcleos: 2
- RAM: 4 GB
- Autoscaling: OFF


In [0]:
{
  "username": "silviaposso01",
  "key": "KGAT_58c30ebb3b9e63f98ea4c7a6d7f1e930"
}

    

'4.1.0'

In [0]:
# Instalar la librería Kaggle
!pip install kaggle


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Crear carpeta de configuración
!mkdir ~/.kaggle


In [0]:
import os

os.environ['KAGGLE_USERNAME'] = "silviaposso01"
os.environ['KAGGLE_KEY'] = "KGAT_7dc15428b8a1ce9c40bbaa8b55ad7b5a"


In [0]:
!kaggle datasets download -d mlg-ulb/creditcardfraud
!unzip creditcardfraud.zip


Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
100%|███████████████████████████████████████| 66.0M/66.0M [00:00<00:00, 132MB/s]

Archive:  creditcardfraud.zip
  inflating: creditcard.csv          


In [0]:
!kaggle datasets download -d mlg-ulb/creditcardfraud -p /dbfs/FileStore/tables/
!unzip /dbfs/FileStore/tables/creditcardfraud.zip -d /dbfs/FileStore/tables/


Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
[Errno 5] Input/output error: '/dbfs/FileStore'
unzip:  cannot find or open /dbfs/FileStore/tables/creditcardfraud.zip, /dbfs/FileStore/tables/creditcardfraud.zip.zip or /dbfs/FileStore/tables/creditcardfraud.zip.ZIP.


In [0]:
df = spark.read.csv("/Workspace/Users/silvia.posso@est.iudigital.edu.co/creditcard.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)


root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: double (nullable = true)
 |-- V28: double (nulla

# Esquema del dataset Credit Card Fraud

| Campo  | Tipo    | Nulabilidad | Descripción                           |
|--------|---------|-------------|---------------------------------------|
| Time   | Double  | Sí          | Tiempo desde la primera transacción   |
| V1-V28 | Double  | Sí          | Variables PCA anonimizadas            |
| Amount | Double  | Sí          | Monto de la transacción               |
| Class  | Integer | Sí          | Etiqueta (0=normal, 1=fraude)         |


In [0]:
# Registrar el DataFrame como tabla temporal
df.createOrReplaceTempView("fraude_tarjetas")


In [0]:
%sql

-- Número total de registros
SELECT COUNT(*) FROM fraude_tarjetas;

-- Distribución de clases (fraude vs normal)
SELECT Class, COUNT(*) 
FROM fraude_tarjetas 
GROUP BY Class;


Class,COUNT(*)
1,492
0,284315


# Validación del dataset Credit Card Fraud

El dataset fue cargado en Spark y registrado como tabla `fraude_tarjetas`.  
Se realizaron las siguientes validaciones:

- **Conteo total de registros**: número de filas en el dataset.  
- **Distribución de clases**: cantidad de transacciones normales (Class=0) vs fraudulentas (Class=1).  
- **Esquema**: columnas Time, V1–V28, Amount y Class, todas numéricas excepto la etiqueta.


In [0]:
#validación en PySpark
# Conteo total
print("Total de registros:", df.count())

# Conteo por clase
df.groupBy("Class").count().show()


Total de registros: 284807
+-----+------+
|Class| count|
+-----+------+
|    1|   492|
|    0|284315|
+-----+------+



In [0]:
%sql
SELECT COUNT(*) FROM fraude_tarjetas;

SELECT Class, COUNT(*) 
FROM fraude_tarjetas 
GROUP BY Class;


Class,COUNT(*)
1,492
0,284315


In [0]:
fraudes = df.filter(df.Class == 1)

print("Total fraudes:", fraudes.count())
fraudes.show(10)


Total fraudes: 492
+------+-------------------+-----------------+------------------+----------------+------------------+-------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+-----------------+--------------------+------------------+-----------------+-------------------+-----------------+-------------------+------------------+-------------------+------------------+-------------------+------------------+------------------+-------------------+------------------+------+-----+
|  Time|                 V1|               V2|                V3|              V4|                V5|                 V6|                V7|                 V8|                V9|               V10|               V11|               V12|               V13|              V14|                 V15|               V16|              V17|                V18|              V19|                V20|               V21|               

In [0]:
fraudes = df.filter(df.Class == 1)
fraudes.write.mode("overwrite").saveAsTable("solo_fraudes")


In [0]:
%sql
DESCRIBE TABLE solo_fraudes;
SELECT COUNT(*) AS total_fraudes FROM solo_fraudes;
SELECT * FROM solo_fraudes LIMIT 10;


Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
156685.0,-0.129777610843349,0.141547455482355,-0.894701869697703,-0.457662244561315,0.81060798093987,-0.504723385101755,1.37358754607199,-0.20947645080755,0.208493953635169,-1.61361833850235,-0.800150416391868,0.501435242117417,1.11801432626547,-1.81638414654661,-0.681282550845059,0.105946450120805,0.716591309767583,0.205284622719977,-0.0513220971157383,0.399447041317156,-0.0326432954176307,-0.246526436107371,0.484107764666265,0.359636576631937,-0.435972450825985,-0.248480311517488,0.0215269254860304,0.109191896780292,187.11,1
156710.0,0.202402312000145,1.1762702245705,0.346379123966988,2.88213779866317,1.40713333279049,-0.504354582435915,1.43853746015701,-0.395603478739171,-1.5551416238588,1.08151398923906,-1.51420473629918,-0.97376404264957,-0.709928478785898,0.141331683969234,-1.47031415230732,0.418268345647957,-0.828192503935611,-0.302239423372689,-1.76235002438458,-0.206238791066218,0.242560469143593,0.841230120611063,-0.370156596962237,-0.0260123796205956,0.49195423236033,0.234576263356619,-0.279788418223114,-0.331933388032899,7.59,1
157207.0,1.17075621526687,2.50103820829712,-4.9861592332247,5.37416005823382,0.997798392058317,-1.25900447846418,-1.23768881403143,0.358425624933301,-2.61248850396066,-3.06472953093127,3.48060170327307,-3.73515295429888,-0.594777601745956,-8.22995199210048,-1.47609403608015,-0.408470612919959,-1.48128347210329,0.347626692232083,-1.16441437126035,0.227618255472955,0.123145442805288,-0.713201110003824,-0.0808679133866204,-0.964310260780621,0.338567587684323,0.068629851804667,0.481587901111279,0.268225854368505,4.97,1
157284.0,-0.242244998983291,4.14718554774299,-5.67234909620488,6.49374059630435,1.59116815595921,-1.60252259512767,-0.950462840251984,0.722902909564455,-4.12850525273272,-3.96322447209244,4.46946731169238,-4.6550713248721,-0.441775588896948,-10.1498129226371,0.612797714121901,-0.288725033933543,1.40450707513401,2.13208137860015,0.707199945685905,0.562029861091792,0.249022570668556,-0.480286305732899,-0.286080216527281,-1.15357475551215,-0.0355711283215471,0.559628209963356,0.40944648264684,0.221048104758881,0.77,1
158638.0,-5.97611932262173,-7.19697963053735,-5.38831591092631,5.104798647287,4.67653286692543,-5.56687006039983,-4.29118042196746,0.876530560032016,-1.07547808735089,-3.27256917385192,1.1682159434835,-2.13473197122041,1.12831321986543,-4.56600986017187,-0.126950406755583,-2.82698551177892,-2.86574970146517,-0.912934238200631,0.421144082740547,3.13633813209484,1.45936905794799,-0.13626168712612,0.848177173598492,-0.269916379777294,-1.09505960437355,-0.710905241951938,0.565846302250925,-1.03410718662341,296.0,1
159844.0,-0.408110816961726,3.13294449795257,-3.09802976350239,5.80389261500645,0.890608919007457,-0.501474323471098,-0.440054283341639,0.591828268527081,-3.26769325879204,-2.22307032799021,0.757063214391669,-3.5018037792943,0.246742224128631,-6.0656218797665,0.339583134189158,-1.00572315400358,0.334315623693923,0.421261362446254,1.24714263088632,0.499567913131525,0.0984823222449585,-0.538374705312947,-0.217988987498016,-1.04265661749863,0.314388820647828,0.54324357594221,0.233851015408581,0.119603460907874,45.51,1
160034.0,-2.3493399641893,1.51260401518904,-2.64749719609623,1.75379247237301,0.406327950335496,-2.18849400496996,-0.686934747748875,-0.547984409288573,-0.0995281132954432,-1.67234597814111,2.17297610055437,-3.10347667138871,0.217560544573622,-6.03440257566619,0.164400705894815,-2.38382603356602,-4.16647877527802,-1.77239694698232,-0.526280794204029,-0.0934214753719124,-0.0885194213942628,-0.595178282040403,0.258148130014773,0.0619011334336728,-0.35418019381022,-1.15267106612603,-0.736072755373029,0.733702732493917,4.9,1
160243.0,-2.78386548658584,1.59682357624614,-2.08484398883683,2.51298558446866,-1.44674856585353,-0.828495614406621,-0.732262192061793,-0.203328860039345,-0.347045718430565,-2.16206139007752,1.96612345703981,-3.1274559

# Tabla derivada: solo_fraudes

Se creó una tabla derivada a partir del dataset original `fraude_tarjetas`, 
filtrando únicamente las transacciones fraudulentas (`Class = 1`).

Esto permite analizar de manera específica las características de las transacciones 
fraudulentas y compararlas con las normales.

Validaciones realizadas:
- Conteo total de fraudes.
- Visualización de las primeras filas.
- Consultas SQL y PySpark equivalentes.


In [0]:
%sql
SELECT * 
FROM solo_fraudes
ORDER BY Amount DESC
LIMIT 10;



Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
122608.0,-2.00345953080582,-7.15904171709445,-4.05097631587393,1.30957974749918,-2.05810158798669,-0.0986209270722274,2.88008272715204,-0.727484046608914,1.4603805509699,-1.53160798206082,-1.39432826167269,-0.220718797789479,-1.53099043146804,1.0752476539262,0.388383209307268,-0.660655352312646,0.0933209955444861,0.335742221574637,0.0575510393537501,3.97321702726744,1.24428677489095,-1.01523228673153,-1.80098486605048,0.657585626965743,-0.435617246752788,-0.894508922176968,-0.39755738695085,0.314261714087509,2125.87,1
9064.0,-3.49910753739178,0.258555160773118,-4.48955807274398,4.85389435139681,-6.97452154477167,3.62838209065084,5.43127092073166,-1.94673371078472,-0.775680092900109,-1.9877731876087,4.69039566550924,-6.9980424321007,1.45401198637847,-3.73802333366749,0.31774206255503,-2.01354268083989,-5.1361351026702,-1.18382211654359,1.66339401354543,-3.04262575715977,-1.05236825623151,0.204816873569406,-2.11900744010783,0.170278608305444,-0.393844118118335,0.296367194488607,1.98591321801306,-0.900451638402827,1809.68,1
154278.0,-1.60021129907252,-3.48813018118561,-6.45930280667571,3.24681566349703,-1.61460769860713,-1.26037454854632,0.288223321921388,-0.0489639910109505,-0.734974617920653,-4.44148408070017,2.94437484242405,-3.80546898374331,-2.10222683492697,-6.1061832734711,-0.641736399701777,-1.55596289626148,-2.08406707082134,0.394247352869229,0.0833797227600835,3.18935489592753,1.19117480721903,-0.96714144500067,-1.46342052446637,-0.624230851495284,-0.176461532068143,0.400348351070197,0.152946864261687,0.477774924576796,1504.93,1
62467.0,-5.34466537543542,-0.285760408166142,-3.83561578178611,5.33704802681048,-7.60990870021268,3.8746680317393,1.28962992438915,0.201741916806601,-3.00353243733788,-3.9905513318108,4.98601380361468,-6.11638322304304,0.0423244061770454,-6.04339314434942,1.82140103177852,-4.7454624675489,-10.0756455907271,-3.6045960155148,1.43529381015392,-1.56216235780532,0.276010822180054,1.34204455689584,-1.01657869464831,-0.0713612973880671,-0.335869334726878,0.441044176004645,1.52061262049821,-1.11593680318009,1402.16,1
59011.0,-2.3269223698887,-3.34843873033788,-3.51340796089061,3.17505971028128,-2.81513703435586,-0.203362788049123,-0.892144430418318,0.333225702638422,-0.8020052289805,-4.35068531246963,3.06424584128211,-2.71873063246571,0.0687877027069899,-5.58687312563725,-0.966076477695954,-2.50204941657576,-4.46049463214891,-0.870525745045081,0.595629103995644,3.20917094389277,1.22664839880046,-0.695902072928095,-1.47849021051453,-0.0615525997443275,0.236155012156618,0.531910616094765,0.302324354380578,0.536375347733134,1389.56,1
65385.0,-2.92382666427262,1.52483716405302,-3.01875803326242,3.28929147538496,-5.75554237361289,2.21827630310359,-0.509995431790455,-3.56944356866613,-1.01659168855188,-4.32053577559936,1.27720226474415,-3.70174966668488,-0.97186959353265,-4.85777667212588,0.0906055221312176,-2.80150199048762,-4.18680753704806,-1.64840609367465,1.1764455977519,-0.447039406719925,-0.511656805196235,-0.12272408639124,-4.28863899895194,0.563796657800838,-0.949451350280346,-0.204532177495445,1.51020648735902,-0.324706012560446,1354.25,1
133184.0,-1.21268170063531,-2.48482353043406,-6.39718581511927,3.67056244781436,-0.863375060980971,-1.85585473063724,1.01773157976221,-0.54470377516891,-1.70337804998263,-3.73965947930309,1.73812401383737,-2.84444933585435,0.765863961245672,-4.79973713485762,-0.0113354116437183,-2.69316808210374,-3.16695515760391,-1.06780011996093,-0.559132202274595,2.90837394592783,1.39687206274272,0.0920728715219574,-1.49288249863592,-0.204227396265286,0.532510949154644,-0.293871104406878,0.212663059827185,0.431094707976035,1335.0,1
18088.0,-12.2240206243564,3.85415032971366,-12.4667657248211,9.64831072711484,-2.72696132147655,-4.44561038185083,-21.9228110340575,0.320792273696859,-4.43316165783721,-11.2014000859835,9.32879925655782,-13.1049334662012,0.888480787

# Consultas SQL sobre la tabla solo_fraudes

1. **Conteo total de fraudes**  
   ```sql
   SELECT COUNT(*) AS total_fraudes FROM solo_fraudes;


In [0]:
# Conteo por clase
df.groupBy("Class").count().show()

# Promedio por clase
df.groupBy("Class").avg("Amount").show()

# Top 10 fraudes más altos
fraudes.orderBy(fraudes.Amount.desc()).show(10)


+-----+------+
|Class| count|
+-----+------+
|    1|   492|
|    0|284315|
+-----+------+

+-----+------------------+
|Class|       avg(Amount)|
+-----+------------------+
|    1|122.21132113821139|
|    0| 88.29102242231878|
+-----+------------------+

+--------+-------------------+------------------+-----------------+-----------------+------------------+-------------------+------------------+-------------------+------------------+-----------------+-----------------+------------------+------------------+-----------------+-------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+-------------------+--------------------+------------------+--------------------+-------+-----+
|    Time|                 V1|                V2|               V3|               V4|                V5|                 V6|                V7|                 V8|                V9| 

# Consultas PySpark sobre fraudes

En esta sección se realizan validaciones y consultas directamente con PySpark 
sobre el DataFrame `fraudes`, que contiene únicamente las transacciones fraudulentas (`Class = 1`).

1. **Conteo total de fraudes**
   ```python
   fraudes.count()
